<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 2 (Martes) — Clasificación 1 — KNN (k vecinos más cercanos)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🎵 KNN: clasificar el género musical de una canción

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Clasificar automáticamente el género de una canción a partir de su audio. |
| 🔍 Comprensión de los datos | Descargamos 900 fragmentos de audio (9 géneros × 100) y los resumimos con 13 coeficientes MFCC. |
| 🧹 Preparación de los datos | Construimos la tabla de características (m1…m13 + género) y separamos entrenamiento/prueba (80/20). |
| 🤖 Modelado | Entrenamos KNN con k = 9 y buscamos el mejor `k` entre 1 y 49. |
| 📊 Evaluación | Matriz de confusión e informe de clasificación por género. |
| 🚀 Despliegue y conclusiones | Predecimos el género de una canción nueva (archivo mp3 → wav → MFCC → KNN). |

> 🎯 **Analogía:** un DJ con muy buen oído: escucha una canción nueva y la ubica junto a las canciones más parecidas que ya conoce.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 👉 **2** | Martes | **Clasificación 1: árboles de decisión, bosques aleatorios y KNN** |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

# Clasificar los géneros musicales de una lista de audios

k-nearest neighbors, abreviado k-nn es un método de clasificación supervisada que sirve para estimar la función de densidad de las predictoras x por cada clase.
- KNN

### Autores
   * Efrén Jiménez

---
## 🎯 Fase 1 · Comprensión del negocio

Clasificar automáticamente el género de una canción a partir de su audio.

📚 Lectura recomendada: [4]

## Análisis del Problema

Con varios sitios que ofrecen descargas de música, categorizar la música descargada se convierte en un desafío y muchas veces también es bastante frustrante. No se puede pensar en escuchar todos los archivos MP3 antes de colocarlos en el contenedor de almacenamiento correcto. Si podemos entrenar la máquina para clasificar estos archivos para permitirnos agruparlos en ciertas categorías predefinidas

---
## 🔍 Fase 2 · Comprensión de los datos — obtener los audios y extraer características

Descargamos 900 fragmentos de audio (9 géneros × 100) y los resumimos con 13 coeficientes MFCC.

📚 Lectura recomendada: [5] [6]

### 🧪 Teoría rápida: ¿cómo clasifica KNN?

KNN guarda todos los ejemplos de entrenamiento. Para un ejemplo nuevo: (1) calcula su distancia a todos los demás, (2) toma los **k más cercanos** y (3) asigna la clase más votada [3].

![Idea de KNN](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_23.png)
*Figura: el ejemplo "?" se clasifica según sus vecinos (3 triángulos y 1 círculo y 1 cruz ⇒ triángulo). [10]*

**Del audio a números:** una canción es una señal; se divide en ventanas de 20 ms y de cada una se calculan los **MFCC** (*Mel-Frequency Cepstral Coefficients*), que resumen el timbre del sonido [5]. Aquí usamos la **media** de cada uno de los 13 coeficientes como el "perfil sonoro" de la canción.

### Librerías

In [ ]:
%pip install -q python_speech_features pydub "audioop-lts; python_version >= '3.13'"

In [ ]:
import numpy as np
import os
import pickle
import random
import pandas as pd
import sklearn
import scipy.io.wavfile as wav
from os import path
from pydub import AudioSegment
from python_speech_features import mfcc
from sklearn.metrics import classification_report
from collections import defaultdict
from sklearn.metrics import accuracy_score
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import os, urllib.request, zipfile

if not os.path.isdir('music-genre-master/Dataset'):
    print('Descargando el conjunto de audios (≈ 2 GB; tarda unos minutos)...')
    urllib.request.urlretrieve('https://github.com/abcom-mltutorials/music-genre/archive/master.zip', 'master.zip')
    with zipfile.ZipFile('master.zip') as z:
        z.extractall('.')
    print('Listo.')
else:
    print('El conjunto de audios ya está descargado.')

In [ ]:
def feature_extraction(file):
    """Devuelve los 13 MFCC promedio de un archivo wav."""
    sampleRate, data = wav.read(file)
    mfcc_feature = mfcc(data, sampleRate, winlen=0.020, appendEnergy=False)
    return list(mfcc_feature.mean(0))

In [ ]:
directory = "music-genre-master/Dataset"
filelist = []
for path, subdirs, files in os.walk(directory):
    for file in files:
        if file.lower().endswith('.wav'):
            filelist.append(os.path.join(path, file))
print("Archivos de audio encontrados:", len(filelist))

In [ ]:
datasetDirectory = "music-genre-master/Dataset/"

# Un género por carpeta; se ordenan para que la etiqueta numérica sea siempre la misma
generos = sorted(os.listdir(datasetDirectory))
print("Géneros:", generos)

filas = []
for etiqueta, genero in enumerate(generos, start=1):
    for archivo in sorted(os.listdir(datasetDirectory + genero)):
        filas.append(feature_extraction(datasetDirectory + genero + "/" + archivo) + [etiqueta])
print("Canciones procesadas:", len(filas))

In [ ]:
filas[0]   # 13 MFCC promedio + etiqueta del género

In [ ]:
print(len(filas), 'canciones;', len(filas[0]), 'valores por canción (13 MFCC + etiqueta)')

In [ ]:
df = pd.DataFrame(filas, columns=['m1','m2','m3','m4','m5','m6','m7',
                                  'm8','m9','m10','m11','m12','m13','target'])
df['target'] = df['target'].astype(int)

In [ ]:
df

---
## 🔍 Fase 2 · Comprensión de los datos — descripción y exploración

Descargamos 900 fragmentos de audio (9 géneros × 100) y los resumimos con 13 coeficientes MFCC.

## Entendimiento de los Datos

El total es de 900 registro con 14 columnas dentro del set de datos

- m1 a m13 : variable de tipo cuantitativa continua
- target      : variable de tipo cuantitativa nominal   

In [ ]:
df


In [ ]:
#Describir dataset
df.describe()

## Exploración de los Datos

In [ ]:
plt.figure(1 , figsize = (15 , 5))
sns.countplot(y = 'target' , data = df)
plt.show()

---
## 🧹 Fase 3 · Preparación de los datos

Construimos la tabla de características (m1…m13 + género) y separamos entrenamiento/prueba (80/20).

📚 Lectura recomendada: [8]

In [ ]:
x1=df[['m1','m2','m3','m4','m5','m6','m7','m8','m9','m10','m11','m12','m13']]
x1.shape

Y = df[['target']]
Y.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x1, Y,
                                                   test_size=0.2,
                                                   random_state=0)

---
## 🤖 Fase 4 · Modelado

Entrenamos KNN con k = 9 y buscamos el mejor `k` entre 1 y 49.

📚 Lectura recomendada: [3] [7]

## Modelo de Machine Learning

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier(n_neighbors=9)

In [ ]:
neigh.fit(X_train,y_train.values.ravel())

In [ ]:
predicted_value = neigh.predict(X_test)
predicted_value

In [ ]:
#will first check which is the best k
Ks =50
mean_acc = np.zeros((Ks-1))
std_acc = np.zeros((Ks-1))
#ConfustionMx = [];
for n in range(1,Ks):
    #Train Model and Predict
    neigh = KNeighborsClassifier(n_neighbors = n,p=2).fit(X_train,y_train.values.ravel())
    yhat=neigh.predict(X_test)
    #yhat = knn_predictions(X_train,y_train,X_test,n)
    mean_acc[n-1] = metrics.accuracy_score(y_test, yhat)
    std_acc[n-1] = np.std(yhat==y_test.values.ravel())/np.sqrt(yhat.shape[0])

print( "Mejor exactitud:", np.round(mean_acc.max()*100,2), "% con k =", mean_acc.argmax()+1)

In [ ]:
plt.plot(range(1,Ks),mean_acc,'g')
plt.fill_between(range(1,Ks),mean_acc - 1 * std_acc,mean_acc + 1 * std_acc, alpha=0.05)
plt.legend(('Accuracy ', '± 1 desv. estándar'))
plt.ylabel('Accuracy ')
plt.xlabel('Number of Neighbors (k)')
plt.tight_layout()
plt.show()

---
## 📊 Fase 5 · Evaluación

Matriz de confusión e informe de clasificación por género.

📚 Lectura recomendada: [9]

## Evaluación

In [ ]:
y_test.groupby("target").target.count()

In [ ]:
results = {i: g for i, g in enumerate(generos, start=1)}
listCategories = generos
results

In [ ]:
cm = confusion_matrix(y_test, predicted_value)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=generos).plot(ax=ax, xticks_rotation=45)
plt.show()

In [ ]:
print(classification_report(y_test, predicted_value, target_names=generos))

---
## 🚀 Fase 6 · Despliegue y conclusiones

Predecimos el género de una canción nueva (archivo mp3 → wav → MFCC → KNN).

📚 Lectura recomendada: [5]

In [ ]:
audio_file = "music-genre-master/new_audio_file.wav"
audio_feature = feature_extraction(audio_file)
audio_feature

In [ ]:

neigh.predict([audio_feature])

In [ ]:

pred_audio=neigh.predict([audio_feature])
results[int(pred_audio[0])]

In [ ]:
import urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/abcom-mltutorials/music/master/bhatiyar.mp3', 'bhatiyar.mp3')
src = "bhatiyar.mp3"
dst = "test.wav"


In [ ]:
sound = AudioSegment.from_mp3(src)
sound.export(dst, format="wav")

In [ ]:
audio_file = "test.wav"
audio_feature = feature_extraction(audio_file)

In [ ]:
neigh.predict([audio_feature])

In [ ]:
pred_audio=neigh.predict([audio_feature])
results[int(pred_audio[0])]

---
## 🧪 Refuerzo: escalar las variables y elegir k con validación cruzada

KNN decide por **distancias**; si una variable tiene valores mucho más grandes que otras, domina la distancia. Por eso se **estandarizan** las variables (media 0, desviación 1) dentro de un `Pipeline`, de modo que el escalado se aprenda solo con los datos de entrenamiento y no haya fuga de información [8]. Además de elegir `k`, se pueden ponderar los vecinos por su cercanía (`weights='distance'`) y cambiar la métrica (euclidiana o Manhattan) [3].

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

X = df[[f'm{i}' for i in range(1, 14)]]
y = df['target'].astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

filas = []
for escalar in (False, True):
    for pesos in ('uniform', 'distance'):
        for metrica in ('euclidean', 'manhattan'):
            pasos = ([StandardScaler()] if escalar else []) + [KNeighborsClassifier(n_neighbors=9, weights=pesos, metric=metrica)]
            acc = cross_val_score(make_pipeline(*pasos), X, y, cv=cv).mean()
            filas.append({'escalado': escalar, 'pesos': pesos, 'métrica': metrica, 'exactitud (5-fold)': round(acc, 3)})
pd.DataFrame(filas).sort_values('exactitud (5-fold)', ascending=False)

### ✍️ Ejercicios
1. ¿Mejora la exactitud al estandarizar los 13 coeficientes MFCC? ¿Por qué podría ayudar poco si todas ya están en escalas parecidas?
2. Repita la comparación con `n_neighbors` = 3, 15 y 31 y grafique exactitud vs. `k`.
3. Con la matriz de confusión: ¿qué pares de géneros se confunden más (p. ej. *rock* y *metal*)? Proponga una razón acústica.

---
## ✅ Conclusiones y siguientes pasos

Con solo 13 números por canción, KNN logra separar géneros con timbres distintos (*clásica*, *metal*), pero confunde géneros parecidos (*rock*, *country*, *blues*). Cambiar el modelo (SVM, bosque aleatorio, red neuronal) o añadir más características (tempo, contraste espectral) suele mejorar el resultado [4].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [11].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory, 13*(1), 21–27. https://doi.org/10.1109/TIT.1967.1053964

**[4]** Tzanetakis, G., & Cook, P. (2002). Musical genre classification of audio signals. *IEEE Transactions on Speech and Audio Processing, 10*(5), 293–302. https://doi.org/10.1109/TSA.2002.800560

**[5]** Davis, S., & Mermelstein, P. (1980). Comparison of parametric representations for monosyllabic word recognition in continuously spoken sentences. *IEEE Transactions on Acoustics, Speech, and Signal Processing, 28*(4), 357–366. https://doi.org/10.1109/TASSP.1980.1163420

**[6]** abcom-mltutorials. (s. f.). *music-genre* [Repositorio con 900 fragmentos de audio en 9 géneros]. GitHub. https://github.com/abcom-mltutorials/music-genre

**[7]** Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2.ª ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

**[8]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[9]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[10]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[11]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a